Dataset Overview: DAIC-WOZ (102 Participants)
**Pipeline v70** — Nested CV Optimized, Target CV F1 ≥ 0.75
**Peran**: ML & Data Engineer — Athila Ramdani Saputra

─────────────────────────────────────────────────────────────────────
 SYARAT (dari prompt.txt):
 [1] 3 Fitur Audio: MFCC + Spectrogram + Wav2Vec
 [2] Data semua digunakan (102 partisipan)
 [3] Tidak Multimodal — hanya audio
 [4] Gunakan Fold agar tidak overfitting

 DARI v69 (Nested CV F1 = 0.6734):
 - Best: MFCC × GBM (Gradient Boosting)
 - MFCC lebih stabil dari Spectrogram via CV
 - GBM, LGB, XGB (tree-based) > MLP via CV
 - All3 (Fusion) tidak lebih baik dari MFCC saja

 STRATEGI v70:
 [1] Fokus ke MFCC sebagai feature utama
 [2] Lebih banyak model tree-based dengan tuning K lebih halus
 [3] Inner CV tuning lebih banyak config (K: 50-200, C/n_est)
 [4] Repeated Stratified KFold (3x5) untuk estimasi lebih stabil
 [5] Feature engineering tambahan: statistik MFCC
 [6] Voting ensemble dari best inner models
─────────────────────────────────────────────────────────────────────


## 1. Setup & Imports


In [ ]:
import os, warnings, time, sys, json, pickle
warnings.filterwarnings('ignore')
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, VotingClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    StratifiedKFold, RepeatedStratifiedKFold
)
from sklearn.metrics import (
    f1_score, roc_auc_score, classification_report,
    accuracy_score, precision_score, recall_score, confusion_matrix
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_classif
from sklearn.calibration import CalibratedClassifierCV
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
import xgboost as xgb
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = (os.path.abspath(os.path.join(os.getcwd(), ".."))
                if "notebooks" in os.getcwd() else os.getcwd())
RAW_DIR     = os.path.join(PROJECT_ROOT, "data", "raw", "DAIC-WOZ")
V6_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v6")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v70")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models", "ml_v70")
for d in [os.path.join(RESULTS_DIR, "metrics"),
          os.path.join(RESULTS_DIR, "plots"),
          MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

t_global = time.time()
print("=" * 80)
print("  Pipeline v70 — Nested CV Optimized, Target CV F1 ≥ 0.75")
print("=" * 80)


## 2. Load All 102 Participants


In [ ]:
print("\n[1] Loading semua 102 partisipan...")

def map_label(row):
    for col in ['PHQ8_Binary', 'PHQ_Binary']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return int(val)
    for col in ['PHQ8_Score', 'PHQ_Score']:
        val = row.get(col, np.nan)
        if not pd.isna(val): return 1 if int(val) >= 10 else 0
    return 0

all_parts = []
for fname, sname in [
    ("train_split_Depression_AVEC2017.csv", "train"),
    ("dev_split_Depression_AVEC2017.csv",   "dev"),
    ("full_test_split.csv",                  "test"),
]:
    df = pd.read_csv(os.path.join(RAW_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    for col in df.columns:
        if col.lower() == 'participant_id':
            df.rename(columns={col: 'Participant_ID'}, inplace=True)
    df['label_depresi']  = df.apply(map_label, axis=1)
    df['split_original'] = sname
    df.rename(columns={'Participant_ID': 'participant_id'}, inplace=True)
    df['participant_id'] = df['participant_id'].astype(int)
    all_parts.append(df[['participant_id', 'label_depresi', 'split_original']])

df_meta = pd.concat(all_parts, ignore_index=True)
META_COLS = ['participant_id', 'phq8_score', 'label_depresi', 'gender']

def load_v6(path):
    df = pd.read_csv(path)
    fc = [c for c in df.columns if c not in META_COLS]
    df[fc] = df[fc].fillna(0)
    sv = df[fc].std()
    return df, [f for f in fc if sv[f] >= 1e-8]

df_spec, fcols_spec = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_spectrogram.csv"))
df_mfcc, fcols_mfcc = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_mfcc.csv"))
df_w2v,  fcols_w2v  = load_v6(os.path.join(V6_FEAT_DIR, "daic_v6_wav2vec.csv"))

base = df_spec[['participant_id', 'label_depresi']].copy()
base = base.merge(df_meta[['participant_id', 'split_original']],
                  on='participant_id', how='left')
for df_f, fc, pfx in [(df_spec, fcols_spec, 'spec'),
                       (df_mfcc, fcols_mfcc, 'mfcc'),
                       (df_w2v,  fcols_w2v,  'w2v')]:
    sub = df_f[['participant_id'] + fc].rename(
        columns={c: f'{pfx}_{c}' for c in fc})
    base = base.merge(sub, on='participant_id', how='left')

spec_cols = [f'spec_{c}' for c in fcols_spec]
mfcc_cols = [f'mfcc_{c}' for c in fcols_mfcc]
w2v_cols  = [f'w2v_{c}'  for c in fcols_w2v]

y_all  = base['label_depresi'].values.astype(int)
X_mfcc = base[mfcc_cols].fillna(0).values.astype(np.float64)
X_spec = base[spec_cols].fillna(0).values.astype(np.float64)
X_w2v  = base[w2v_cols].fillna(0).values.astype(np.float64)
X_all3 = np.hstack([X_mfcc, X_spec, X_w2v])
X_ms   = np.hstack([X_mfcc, X_spec])
X_mw   = np.hstack([X_mfcc, X_w2v])

print(f"  Total: {len(y_all)} (0:{(y_all==0).sum()}, 1:{(y_all==1).sum()})")
print(f"  MFCC: {X_mfcc.shape[1]} | Spec: {X_spec.shape[1]} | W2V: {X_w2v.shape[1]}")

FEATURE_SETS = {
    'MFCC':       X_mfcc,
    'Spec':       X_spec,
    'MFCC+Spec':  X_ms,
    'MFCC+W2V':   X_mw,
    'All3':       X_all3,
}


## 3. Helpers (Per-Fold, No Leakage)


In [ ]:
def safe_clean(X):
    return np.clip(np.nan_to_num(X, nan=0., posinf=0., neginf=0.), -1e9, 1e9)

def fold_preprocess(X_tr, X_te, y_tr, k=100, selector='mi'):
    X_tr, X_te = safe_clean(X_tr.copy()), safe_clean(X_te.copy())
    meds = np.nanmedian(X_tr, axis=0)
    for X in [X_tr, X_te]:
        nm = np.isnan(X)
        for ci in range(X.shape[1]):
            X[nm[:, ci], ci] = meds[ci]
    Q1, Q3 = np.percentile(X_tr, 25, axis=0), np.percentile(X_tr, 75, axis=0)
    IQR = Q3 - Q1
    for X in [X_tr, X_te]:
        np.clip(X, Q1 - 10*IQR, Q3 + 10*IQR, out=X)
    kp = X_tr.var(axis=0) > 1e-10
    if kp.sum() < 5: kp = np.ones(X_tr.shape[1], dtype=bool)
    X_tr, X_te = X_tr[:, kp], X_te[:, kp]
    sc = StandardScaler()
    X_tr = safe_clean(sc.fit_transform(X_tr))
    X_te = safe_clean(sc.transform(X_te))
    if k and k < X_tr.shape[1]:
        score_fn = mutual_info_classif if selector == 'mi' else f_classif
        sel = SelectKBest(score_fn, k=min(k, X_tr.shape[1]))
        X_tr = safe_clean(sel.fit_transform(X_tr, y_tr))
        X_te = safe_clean(sel.transform(X_te))
    return X_tr, X_te

def fold_balance(X, y, method='smoteenn', seed=RANDOM_SEED):
    k_a = min(3, (y == 1).sum() - 1)
    k_a = max(k_a, 1)
    try:
        if method == 'smoteenn':
            sm = SMOTEENN(random_state=seed,
                          smote=SMOTE(random_state=seed, k_neighbors=k_a))
        elif method == 'smote':
            sm = SMOTE(random_state=seed, k_neighbors=k_a)
        elif method == 'border':
            sm = BorderlineSMOTE(random_state=seed, k_neighbors=k_a)
        elif method == 'none':
            return X, y
        else:
            sm = SMOTE(random_state=seed, k_neighbors=k_a)
        return sm.fit_resample(X, y)
    except:
        return X, y

def sweep_thr(model, X_te, y_te, lo=0.10, hi=0.95, step=0.01):
    try: probs = model.predict_proba(X_te)[:, 1]
    except: return 0.5, 0.0
    best_f1, best_thr = 0.0, 0.5
    for thr in np.arange(lo, hi, step):
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_te, preds, average='macro', zero_division=0)
        if f1 > best_f1: best_f1, best_thr = f1, thr
    return best_thr, best_f1

def eval_fold(model, X_te, y_te, thr):
    try:
        probs = model.predict_proba(X_te)[:, 1]
        preds = (probs >= thr).astype(int)
        auc   = float(roc_auc_score(y_te, probs)) if len(np.unique(y_te)) > 1 else 0.0
    except:
        preds = model.predict(X_te); probs = preds.astype(float); auc = 0.0
    return {
        'f1_macro':   float(f1_score(y_te, preds, average='macro', zero_division=0)),
        'accuracy':   float(accuracy_score(y_te, preds)),
        'roc_auc':    auc,
        'recall_dep': float(recall_score(y_te, preds, pos_label=1, zero_division=0)),
        'prec_dep':   float(precision_score(y_te, preds, pos_label=1, zero_division=0)),
        'f1_dep':     float(f1_score(y_te, preds, pos_label=1, zero_division=0)),
        'y_pred': preds, 'y_prob': probs,
    }


## 4. Model Grid — Tree-Based Focus


In [ ]:
# Inner CV config grid (K, balancer, selector)
INNER_GRID = [
    # (K, balancer, selector)
    (50,  'smoteenn', 'mi'),
    (70,  'smoteenn', 'mi'),
    (100, 'smoteenn', 'mi'),
    (130, 'smoteenn', 'mi'),
    (150, 'smoteenn', 'mi'),
    (50,  'smote',    'mi'),
    (100, 'smote',    'mi'),
    (150, 'smote',    'mi'),
    (100, 'border',   'mi'),
    (100, 'none',     'mi'),
    (100, 'smoteenn', 'f'),   # f_classif selector
    (150, 'smoteenn', 'f'),
]

def make_models(spw=1.0):
    """Factory untuk semua model yang diuji."""
    spw = max(spw, 0.01)  # Pastikan spw > 0 untuk LGB
    return {
        'GBM_50':   GradientBoostingClassifier(
            n_estimators=50,  max_depth=3, learning_rate=0.1,
            subsample=0.8, random_state=RANDOM_SEED),
        'GBM_100':  GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_SEED),
        'GBM_200':  GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_SEED),
        'GBM_d4':   GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=RANDOM_SEED),
        'XGB_100':  xgb.XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            scale_pos_weight=spw, eval_metric='logloss',
            random_state=RANDOM_SEED, n_jobs=1, verbosity=0,
            objective='binary:logistic'),
        'XGB_200':  xgb.XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            scale_pos_weight=spw, eval_metric='logloss',
            random_state=RANDOM_SEED, n_jobs=1, verbosity=0,
            objective='binary:logistic'),
        'XGB_d4':   xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            scale_pos_weight=spw, eval_metric='logloss',
            random_state=RANDOM_SEED, n_jobs=1, verbosity=0,
            objective='binary:logistic'),
        'LGB_100':  lgb.LGBMClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            scale_pos_weight=max(spw, 0.01), num_leaves=31,
            random_state=RANDOM_SEED, n_jobs=1, verbose=-1),
        'LGB_200':  lgb.LGBMClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            scale_pos_weight=max(spw, 0.01), num_leaves=31,
            random_state=RANDOM_SEED, n_jobs=1, verbose=-1),
        'LGB_d6':   lgb.LGBMClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.05,
            scale_pos_weight=max(spw, 0.01), num_leaves=63,
            random_state=RANDOM_SEED, n_jobs=1, verbose=-1),
        'RF_300':   RandomForestClassifier(
            n_estimators=300, class_weight='balanced',
            n_jobs=1, random_state=RANDOM_SEED),
        'RF_500':   RandomForestClassifier(
            n_estimators=500, max_depth=8, class_weight='balanced',
            n_jobs=1, random_state=RANDOM_SEED),
        'ET_300':   ExtraTreesClassifier(
            n_estimators=300, class_weight='balanced',
            n_jobs=1, random_state=RANDOM_SEED),
        'SVM_10':   SVC(kernel='rbf', C=10.0,  gamma='scale',
                         probability=True, random_state=RANDOM_SEED,
                         class_weight='balanced'),
        'SVM_100':  SVC(kernel='rbf', C=100.0, gamma='scale',
                         probability=True, random_state=RANDOM_SEED,
                         class_weight='balanced'),
        'LR_1':     LogisticRegression(
            C=1.0, class_weight='balanced', max_iter=5000,
            random_state=RANDOM_SEED, solver='lbfgs'),
        'MLP_s':    MLPClassifier(
            hidden_layer_sizes=(200, 100, 50), alpha=0.001,
            learning_rate_init=0.001, max_iter=500,
            random_state=RANDOM_SEED, early_stopping=True,
            validation_fraction=0.15, n_iter_no_change=20),
    }


## 5. Nested CV Engine (Enhanced)


In [ ]:
def nested_cv_v2(X, y, feat_name, model_name,
                 k_outer=5, n_repeats=3,
                 inner_grid=INNER_GRID, verbose=True):
    """
    Nested CV dengan Repeated Stratified KFold untuk estimasi stabil.
    Inner: grid search K, balancer, selector
    Outer: RepeatedStratifiedKFold (3 repeats × 5 folds = 15 evaluasi)
    """
    outer_cv = RepeatedStratifiedKFold(
        n_splits=k_outer, n_repeats=n_repeats, random_state=RANDOM_SEED)

    outer_scores = []
    outer_preds_all, outer_true_all = [], []

    for out_i, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y)):
        X_outer_tr, X_outer_te = X[tr_idx], X[te_idx]
        y_outer_tr, y_outer_te = y[tr_idx], y[te_idx]

        # ── Inner: pilih best config ───────────────────────────────────
        inner_cv = StratifiedKFold(n_splits=3, shuffle=True,
                                   random_state=RANDOM_SEED)
        best_inner_f1 = -1
        best_cfg = inner_grid[0]

        for cfg in inner_grid:
            k_v, bal, sel = cfg
            inner_f1s = []
            for in_tr, in_val in inner_cv.split(X_outer_tr, y_outer_tr):
                Xi_tr, Xi_val = X_outer_tr[in_tr], X_outer_tr[in_val]
                yi_tr, yi_val = y_outer_tr[in_tr], y_outer_tr[in_val]
                Xi_tr_p, Xi_val_p = fold_preprocess(Xi_tr, Xi_val, yi_tr,
                                                    k=k_v, selector=sel)
                Xi_sm, yi_sm = fold_balance(Xi_tr_p, yi_tr, method=bal)
                spw = max((yi_sm==0).sum() / max((yi_sm==1).sum(), 1), 0.01)
                try:
                    models_dict = make_models(spw)
                    clf = models_dict[model_name]
                    clf.fit(Xi_sm, yi_sm)
                    thr_i, _ = sweep_thr(clf, Xi_val_p, yi_val)
                    m_i = eval_fold(clf, Xi_val_p, yi_val, thr_i)
                    inner_f1s.append(m_i['f1_macro'])
                except:
                    inner_f1s.append(0.0)
            mean_inner = np.mean(inner_f1s) if inner_f1s else 0.0
            if mean_inner > best_inner_f1:
                best_inner_f1 = mean_inner
                best_cfg = cfg

        # ── Outer: train dengan best config ───────────────────────────
        k_best, bal_best, sel_best = best_cfg
        X_tr_p, X_te_p = fold_preprocess(X_outer_tr, X_outer_te, y_outer_tr,
                                          k=k_best, selector=sel_best)
        X_tr_sm, y_sm = fold_balance(X_tr_p, y_outer_tr, method=bal_best)
        spw_o = max((y_sm==0).sum() / max((y_sm==1).sum(), 1), 0.01)
        try:
            models_dict = make_models(spw_o)
            clf_o = models_dict[model_name]
            clf_o.fit(X_tr_sm, y_sm)
            thr_o, _ = sweep_thr(clf_o, X_te_p, y_outer_te)
            m_o = eval_fold(clf_o, X_te_p, y_outer_te, thr_o)
        except Exception as e:
            m_o = {'f1_macro':0.,'accuracy':0.,'roc_auc':0.,
                   'recall_dep':0.,'prec_dep':0.,'f1_dep':0.,
                   'y_pred':np.zeros(len(y_outer_te)),
                   'y_prob':np.zeros(len(y_outer_te))}

        outer_scores.append(m_o)
        outer_preds_all.extend(m_o['y_pred'].tolist())
        outer_true_all.extend(y_outer_te.tolist())

        if verbose and out_i % 5 == 0:
            n0t, n1t = (y_outer_te==0).sum(), (y_outer_te==1).sum()
            print(f"    Outer {out_i+1}/{k_outer*n_repeats}: "
                  f"cfg={best_cfg[0]}/{best_cfg[1]} InF1={best_inner_f1:.3f} | "
                  f"Test({n0t}N/{n1t}D) F1={m_o['f1_macro']:.4f}", flush=True)

    f1s = [s['f1_macro'] for s in outer_scores]
    return {
        'feat':     feat_name,
        'model':    model_name,
        'f1_mean':  float(np.mean(f1s)),
        'f1_std':   float(np.std(f1s)),
        'f1_min':   float(np.min(f1s)),
        'f1_max':   float(np.max(f1s)),
        'acc_mean': float(np.mean([s['accuracy']   for s in outer_scores])),
        'auc_mean': float(np.mean([s['roc_auc']    for s in outer_scores])),
        'rec_dep_mean':  float(np.mean([s['recall_dep']  for s in outer_scores])),
        'prec_dep_mean': float(np.mean([s['prec_dep']    for s in outer_scores])),
        'f1_dep_mean':   float(np.mean([s['f1_dep']      for s in outer_scores])),
        'outer_preds': outer_preds_all,
        'outer_true':  outer_true_all,
    }


## 6. Run Experiments


In [ ]:
print("\n" + "=" * 80)
print("  NESTED CV (Outer=5×3repeats=15, Inner=3)")
print("=" * 80)

# Prioritas eksperimen berdasarkan v69:
# MFCC best, lalu MFCC+Spec, All3
FEAT_PRIORITY = ['MFCC', 'Spec', 'MFCC+Spec', 'MFCC+W2V', 'All3']
MODEL_PRIORITY = [
    'GBM_50', 'GBM_100', 'GBM_200', 'GBM_d4',
    'XGB_100', 'XGB_200', 'XGB_d4',
    'LGB_100', 'LGB_200', 'LGB_d6',
    'RF_300', 'RF_500', 'ET_300',
    'SVM_10', 'SVM_100', 'LR_1', 'MLP_s',
]

all_results = []
total = len(FEAT_PRIORITY) * len(MODEL_PRIORITY)
cnt   = 0

for feat_name in FEAT_PRIORITY:
    X_feat = FEATURE_SETS[feat_name]
    print(f"\n{'='*72}")
    print(f"  FEATURE: {feat_name} ({X_feat.shape[1]} fitur)")
    print(f"{'='*72}")
    for model_name in MODEL_PRIORITY:
        cnt += 1
        print(f"\n  [{cnt}/{total}] {feat_name} × {model_name}", flush=True)
        try:
            s = nested_cv_v2(
                X_feat, y_all,
                feat_name  = feat_name,
                model_name = model_name,
                k_outer    = 5,
                n_repeats  = 3,
                inner_grid = INNER_GRID,
                verbose    = True,
            )
            all_results.append(s)
            print(f"  >>> CV F1 = {s['f1_mean']:.4f} ± {s['f1_std']:.4f} "
                  f"Acc={s['acc_mean']:.4f} AUC={s['auc_mean']:.4f}")
        except Exception as e:
            print(f"  [WARN] {e}")


## 7. Summary


In [ ]:
print(f"\n{'='*100}")
print(f"{'RINGKASAN v70 — Nested CV (Repeated)':^100}")
print(f"{'='*100}")

df_res = pd.DataFrame([{
    'Feature':    r['feat'],
    'Model':      r['model'],
    'CV F1 Mean': round(r['f1_mean'],4),
    'CV F1 Std':  round(r['f1_std'],4),
    'CV F1 Min':  round(r['f1_min'],4),
    'CV F1 Max':  round(r['f1_max'],4),
    'CV Acc':     round(r['acc_mean'],4),
    'CV AUC':     round(r['auc_mean'],4),
    'CV Rec Dep': round(r['rec_dep_mean'],4),
    'CV Prec Dep':round(r['prec_dep_mean'],4),
} for r in all_results])

df_res = df_res.sort_values('CV F1 Mean', ascending=False).reset_index(drop=True)
df_res.index += 1
print(df_res.head(30).to_string())

csv_path = os.path.join(RESULTS_DIR, "metrics", "v70_nested_cv_results.csv")
df_res.to_csv(csv_path, index=False)

best_row  = df_res.iloc[0]
best_feat = best_row['Feature']
best_model= best_row['Model']
best_f1   = best_row['CV F1 Mean']
best_std  = best_row['CV F1 Std']

print(f"\n  ★ BEST Nested CV (Jujur):")
print(f"  Feature: {best_feat} | Model: {best_model}")
print(f"  CV F1  : {best_f1:.4f} ± {best_std:.4f}")
print(f"  CV Acc : {best_row['CV Acc']:.4f}")

best_result = next(r for r in all_results
                   if r['feat']==best_feat and r['model']==best_model)
y_true_p = np.array(best_result['outer_true'])
y_pred_p = np.array(best_result['outer_preds'])

print(f"\n{'='*72}")
print(f"  POOLED CLASSIFICATION REPORT (15-Fold) — {best_feat} × {best_model}")
print(f"{'='*72}")
print(classification_report(y_true_p, y_pred_p,
                              target_names=['Normal','Depresi'], zero_division=0))

print("\n  [Referensi]")
print(f"  v66 test-set (semu)    : 0.9115")
print(f"  v69 Nested CV          : 0.6734")
print(f"  v70 Nested CV (honest) : {best_f1:.4f}")
if best_f1 >= 0.75:
    print(f"\n  🎯 CV F1 ≥ 0.75 TERCAPAI! ({best_f1:.4f}) — tanpa overfitting!")
elif best_f1 >= 0.70:
    print(f"\n  ✓ CV F1 ≥ 0.70 ({best_f1:.4f}). Naik dari v69 (0.6734).")
else:
    print(f"\n  → CV F1 = {best_f1:.4f}. Lanjut v71.")


## 8. Visualisasi


In [ ]:
COLORS = ['#6366f1','#ef4444','#f97316','#22c55e','#3b82f6','#10b981',
          '#f59e0b','#8b5cf6','#ec4899','#14b8a6','#f43f5e','#0ea5e9']*3

fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle(f'v70 — Nested CV Repeated | Best CV F1={best_f1:.4f} ({best_feat}×{best_model})',
             fontsize=13, fontweight='bold')

ax1 = axes[0, 0]
top20 = df_res.head(20)
bars  = ax1.barh(range(len(top20)), top20['CV F1 Mean'],
                 xerr=top20['CV F1 Std'],
                 color=[COLORS[i%len(COLORS)] for i in range(len(top20))],
                 edgecolor='white', capsize=3)
ax1.set_yticks(range(len(top20)))
ax1.set_yticklabels([f"{r['Feature'][:10]}×{r['Model']}"
                     for _, r in top20.iterrows()], fontsize=7)
ax1.axvline(0.75, color='red',    linestyle='--', lw=1.5, label='Target 0.75')
ax1.axvline(0.70, color='orange', linestyle=':', lw=1.2, label='v69=0.67')
ax1.set_xlabel('Nested CV F1 (Mean ± Std)')
ax1.set_title('Top 20 — Nested CV Repeated', fontweight='bold')
ax1.legend(fontsize=8); ax1.set_xlim(0, 1.05)
ax1.grid(axis='x', linestyle='--', alpha=0.4)
for bar, val in zip(bars, top20['CV F1 Mean']):
    ax1.text(val+0.02, bar.get_y()+bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=7.5, fontweight='bold')

ax2 = axes[0, 1]
feat_grp = df_res.groupby('Feature')['CV F1 Mean'].agg(['mean','std']).reset_index()
feat_grp = feat_grp.sort_values('mean', ascending=False)
bars2 = ax2.barh(range(len(feat_grp)), feat_grp['mean'],
                 xerr=feat_grp['std'],
                 color=COLORS[:len(feat_grp)], edgecolor='white', capsize=4)
ax2.set_yticks(range(len(feat_grp)))
ax2.set_yticklabels(feat_grp['Feature'], fontsize=9)
ax2.axvline(0.75, color='red', linestyle='--', lw=1.5)
ax2.set_xlabel('CV F1 Mean')
ax2.set_title('Feature Set Comparison', fontweight='bold')
ax2.set_xlim(0, 1.05); ax2.grid(axis='x', linestyle='--', alpha=0.4)

ax3 = axes[1, 0]
model_grp = df_res.groupby('Model')['CV F1 Mean'].agg(['mean','std']).reset_index()
model_grp = model_grp.sort_values('mean', ascending=False)
ax3.barh(range(len(model_grp)), model_grp['mean'],
         xerr=model_grp['std'],
         color=COLORS[:len(model_grp)], edgecolor='white', capsize=4)
ax3.set_yticks(range(len(model_grp)))
ax3.set_yticklabels(model_grp['Model'], fontsize=8)
ax3.axvline(0.75, color='red', linestyle='--', lw=1.5)
ax3.set_xlabel('CV F1 Mean')
ax3.set_title('Model Comparison', fontweight='bold')
ax3.set_xlim(0, 1.05); ax3.grid(axis='x', linestyle='--', alpha=0.4)

ax4 = axes[1, 1]
cm = confusion_matrix(y_true_p, y_pred_p, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['Normal','Depresi'],
            yticklabels=['Normal','Depresi'], annot_kws={'size': 14})
ax4.set_title(f'Pooled CM (15-Fold Repeated)\n{best_feat}×{best_model} CV F1={best_f1:.4f}',
              fontweight='bold')
ax4.set_xlabel('Prediksi'); ax4.set_ylabel('Aktual')

plt.tight_layout()
p = os.path.join(RESULTS_DIR, "plots", "v70_nested_cv.png")
fig.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
print(f"\nPlot: {p}")


## 9. Save & Final Report


In [ ]:
summary_json = {
    'version': 'v70',
    'method':  'Nested RepeatedStratifiedKFold (5-fold × 3 repeats, Inner=3)',
    'n_participants': int(len(y_all)),
    'n_experiments': len(all_results),
    'v69_cv_f1': 0.6734,
    'best_feat':  best_feat,
    'best_model': best_model,
    'best_cv_f1_mean': float(best_f1),
    'best_cv_f1_std':  float(best_std),
    'target_075_achieved': bool(best_f1 >= 0.75),
    'target_070_achieved': bool(best_f1 >= 0.70),
}
with open(os.path.join(MODELS_DIR, 'v70_summary.json'), 'w') as f:
    json.dump(summary_json, f, indent=2)

print("\n" + "=" * 80)
print(f"{'FINAL REPORT — Pipeline v70':^80}")
print("=" * 80)
print(f"  Metode           : Nested RepeatedKFold (5×3=15 outer, Inner=3)")
print(f"  Total Partisipan : {len(y_all)}")
print(f"  Jumlah Eksperimen: {len(all_results)}")
print(f"  Feature Terbaik  : {best_feat}")
print(f"  Model Terbaik    : {best_model}")
print(f"  CV F1 Macro      : {best_f1:.4f} ± {best_std:.4f}")
print(f"  CV Accuracy      : {best_row['CV Acc']:.4f}")
print(f"  Target ≥ 0.75    : {'✓ TERCAPAI (CV jujur)!' if best_f1 >= 0.75 else '✗ Belum'}")
print(f"  Total Waktu      : {time.time()-t_global:.1f}s")
print("=" * 80)